In [2]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.3233239)

In [6]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730505)

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [10]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [17]:
import numpy as np
X = np.array(vectors)

In [19]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(553), np.float32(0.7629409))

In [22]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [23]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([553, 955,  29, 472, 558])

In [24]:
scores[top5]

array([0.7629409 , 0.757937  , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.757937
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [40]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(
    query_vector, 
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [49]:
from openai import OpenAI
from rag_helper import RAGVector

openai_client=OpenAI()

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [47]:
import sys, os, importlib
print("exe:", sys.executable)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])
try:
    import rag_helper
    print("rag_helper file:", rag_helper.__file__)
    print("names:", [n for n in dir(rag_helper) if not n.startswith('_')])
    importlib.reload(rag_helper)
    print("reloaded; has RAGVector?", hasattr(rag_helper, "RAGVector"))
except Exception as e:
    print("import error:", type(e).__name__, e)

exe: d:\Study\Python\llm-zoomcamp\.venv\Scripts\python.exe
cwd: d:\Study\Python\llm-zoomcamp
sys.path[0]: C:\Users\Sam He\AppData\Roaming\uv\python\cpython-3.14.0-windows-x86_64-none\python314.zip
rag_helper file: d:\Study\Python\llm-zoomcamp\rag_helper.py
names: ['INSTRUCTIONS', 'PROMPT_TEMPLATE', 'RAGBase']
reloaded; has RAGVector? True


In [50]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'